In [20]:
from google import genai
from google.genai import types
import pandas as pd
import google.generativeai as genai
import json
import os.path

import gspread
import openpyxl
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from datetime import datetime, timedelta

def main():
    combined_df = extract_data()
    if combined_df is not None:
        display(combined_df.head(10))
        print(f"Final Data Shape: {combined_df.shape[0]} rows x {combined_df.shape[1]} columns")
        creds = connect_to_sheet()
        update_sheet('1ckgGhsbbHGYfpqXtak_EG79ElY00bTvAADbNySRMa1s', 'Capex All', None, creds, combined_df)

def extract_data():
    files_and_sheets = [
        (r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\1. SP.xlsx", "SUMMARY"),
        (r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\2. ghopo rev.xlsx", "Prognose Spending"),
        #(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\3. SG .xlsx", "SUMMARY"),
        #(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\4. SMBR.xlsx", "SMBR 2026 R1"),
        #(r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\5. ST.xlsx", "4000"),
        (r"C:\Users\mochamad.ilmawan\OneDrive - semenindonesia.com\Renstra\Capex\6. SBI.xlsx", "SUMMARY")
    ]
    
    for path, sheet in files_and_sheets:
        if os.path.exists(path):
            file_name = os.path.basename(path)
            
            last_row = get_last_row(path, sheet)
            print(f"Processing: {file_name} (Sheet: {sheet})... Last row: {last_row}")
            df_raw = pd.read_excel(
                path, 
                sheet_name=sheet, 
                header=[2, 3, 4], 
                nrows=last_row,
            )
            df_raw = df_raw.iloc[:, 1:38]
            
            chunk_size = 30
            all_extracted_chunks = []
            
            for i in range(0, len(df_raw), chunk_size):
                df_chunk = df_raw.iloc[i : i + chunk_size]
                print(f"Sending rows {i} to {i + len(df_chunk)} to Gemini...")
                
                chunk_string = df_chunk.to_string()
                df_data = ai_processing(chunk_string)
                
                if df_data is not None:
                    all_extracted_chunks.append(df_data)
            
            # Combine all the successful chunks back together
            if all_extracted_chunks:
                final_df = pd.concat(all_extracted_chunks, ignore_index=True)
                print(f"Successfully processed sheet! Extracted {len(final_df)} rows.")
            
            

def ai_processing(raw_data_string):
    genai.configure(api_key="AQ.Ab8RN6I7WqbeUlRXPjXm9C75hCweJNo5M8AOkZgJ2HP2atp1WA")
    generation_config = {
        "temperature": 0,
        "top_p": 0.95,
        "response_mime_type": "application/json",
    }
    model = genai.GenerativeModel(
      model_name="gemini-3.5-flash",
      generation_config=generation_config,
    )
    prompt = f"""
            Act as a precise data extraction engine. Extract data metrics from the provided raw Excel dump.

            ### EXTRACTION RULES:
            1. MAPPING: Locate the exact row for each of the 35 metrics listed below.
            2. VALUES: 
               - If a value is missing, null, or "-" in the raw data, return 0.0. 
               - DO NOT use nested dictionaries. Use a flat structure.
               - Fill Month Value with January until Desember for each project.
            4. Fill the missing value in PROJECT CODE / WBS or NAMA CAPEX with "Belum terisi"
            4. FORMAT: Return ONLY a valid JSON list of objects. No conversational text.

            ### METRIC LIST:
            1. Tahun Capex, 2. Item Capex, 3. No. Project / WBS, 4. Entitas, 5. Departemen, 6. Tipe Capex,
            7. NEW / COC, 8. Ruang Lingkup Pengadaan, 9. Rencana Jumlah Pengadaan, 10. Nilai Investasi FS,
            11. Project Charter, 12. Project Manager, 13. Start project, 14. Target Finish project, 
            15. % Plan Progres Fisik (Over all), 16. % Progres Fisik Real (Over all), 
            17. Status Pekerjaan Terakhir, 18. issue, 19. BOL, 20. FS SUBMITED, 21. EVALUATION, 22. FID,
            23. Engineering, 24. Procurement, 25. Construction, 26. Commisioning, 27. BAST, 28. PIR,
            29. Perubahan Ruang Lingkup, 30. Total Nilai Investasi RKAP, 31. Total PR, 32. No PR, 
            33. Total PO, 34. No PO., 37. Spending / GR / SA tahun sebelumnya
            ### REQUIRED JSON STRUCTURE EXAMPLE:
            [
              {{
                "1. Tahun Capex": 2026,
                "2. Item Capex": "IMPLEMENTASI PLANT OPTIMIZER SOFTWARE CM 3, CM 5 DAN CM 9 GHOPO TUBAN ",
                "3. No. Project / WBS": "P6-22270-07-C-12", 
                .....
                "32. No PR": "6200004449, 6200004498", 
                "33. Total PO": 583.925.000,
                "34. No PO": "7400003139, 7400003158",
                "37. Spending / GR / SA tahun sebelumnya": 500.000.000
              }},
              {{
                "1. Tahun Capex": 2026,
                "2. Item Capex": "SP-T REPLACE RBC RH A2J02",
                "3. No. Project / WBS": "P2-26107-03-C-07", 
                .....
                "32. No PR": "6200004449, 6200004498", 
                "33. Total PO": 583.925.000,
                "34. No PO": "7400003139, 7400003158",
                "37. Spending / GR / SA tahun sebelumnya": 0
              }}
            ]

            Raw Data:
            {raw_data_string}
            """# 4. Generate response
    response = model.generate_content(
        prompt,
        request_options={"timeout": 600.0}  # 10 minutes
    )
    json_text = response.text.replace('```json', '').replace('```', '').strip()

    print(f"DEBUG: Response Length: {len(json_text)}") # See if it's hitting a limit
    try:
        data = json.loads(json_text)
        # 5. Convert back to DataFrame
        return pd.DataFrame(data)
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        return None

def get_last_row(file_path, sheet_name):
    wb = openpyxl.load_workbook(file_path, read_only=True)
    ws = wb[sheet_name]

    last_row = 4  # Default to row 4 if no data is found below it
    # Iterate backwards from the maximum sheet row down to row 4
    for r in range(ws.max_row, 3, -1):
        if ws.cell(row=r, column=4).value is not None:  # Column D is the 4th column
            last_row = r
            break
    return last_row

if __name__ == '__main__':
    main()

Processing: 1. SP.xlsx (Sheet: SUMMARY)... Last row: 189
Sending rows 0 to 30 to Gemini...
DEBUG: Response Length: 43160
Sending rows 30 to 60 to Gemini...
DEBUG: Response Length: 42461
Error parsing JSON: Expecting ':' delimiter: line 178 column 19 (char 6884)
Sending rows 60 to 90 to Gemini...


KeyboardInterrupt: 